# Amending dataset using cosine similarity in embedding space

### Identifying edge-cases
This notebook contains the code for identifying edge-cases in our dataset, meaning cases where a text should possibly be coded differently. These edge-cases are identified by calculating the distance between all texts and choosing all pairs that have a cosine similarity higher than 0.95 (corresponding to high semantic similarity) and have different codes.

The notebook starts with the regular imports and loading the dataset that includes the responses and their code.

In [52]:
# Imports
import pickle
import numpy as np
import pandas as pd
import scipy as sc
from sentence_transformers import SentenceTransformer

In [53]:
# Load the data
df = pd.read_excel('data/sources.xlsx')
df

,Response,code
0,Imprecise/faulty measuring tools,L
1,air fluctuations,L
2,Slight differences in experimental setup: tabl...,L
3,The material of ramp and ball and the coeffici...,L
4,Materials issues (ball; ramps are identical),L
...,...,...
2894,imperfections in detection screen,L
2895,differences in the set up between groups (not ...,L
2896,Cosmic ray interference,L
2897,ruler differences,L


Next, we embed the responses using sentence transformer.

In [ ]:
# Embed all the responses
model = SentenceTransformer("mixedbread-ai/mxbai-embed-large-v1")
embeddings = model.encode(df['response'].tolist())
df['embedding'] = embeddings

KeyboardInterrupt: 

Here we calculate the cosine similarity between all pairs and put them into a matrix.

In [10]:
embedding_matrix = np.vstack(df["embedding"].values)

# Calculate the cosine similarity between all the responses
cosine_similarity = sc.spatial.distance.cdist(embedding_matrix, embedding_matrix, 'cosine')

Here we identify pairs with high similarity but different codes.

In [11]:
# If the cosine similarity is above 0.95 and the responses have different codes,
# count it as a duplicate

duplicates = []

for i in range(cosine_similarity.shape[0]):
    temp = []
    for j in range(cosine_similarity.shape[1]):
        if (cosine_similarity[i, j] < 0.15 and df['code'][j] != df['code'][i]):
            temp.append([j, cosine_similarity[i, j]])

    if len(temp) > 0:
        temp = min(temp, key=lambda x: x[1])
        duplicates.append([df['response'][i], df['response'][temp[0]], temp[1], df['code'][i], df['code'][temp[0]], int(i), int(temp[0])])

Further, we save the dataframe into a new dataset for amending. The edge-cases will now be re-evaluated by a qualified qualitative researcher.

In [31]:
df_duplicates = pd.DataFrame(duplicates, columns=['response_1', 'code_1', 'index_1','response_2', 'code_2',  'index_2','cosine_similarity'])

In [13]:
df_duplicates.sort_values(by='cosine_similarity', ascending=True, inplace=True)

In [ ]:
df_duplicates.to_csv('data/duplicates.csv', index=False)

### Amending edge-cases
The dataset has been amended by a qualified qualitative researcher and we get a new dataset with the amended results. It is loaded in the code-cell below. The updated codes can be found in the columns 'updated_code1' and 'updated_code2'.

In [ ]:
# Load the amended data
df_duplicates_amended = pd.read_csv('data/duplicates_amended.csv', sep=";")
df_duplicates_amended

,Unnamed: 0,text1,code1,index1,text2,code2,index2,cosine_distance,updated_code1,updated_code2
0,0,Measurement Error,O,506,Measurement Error,L,40,0.000000,O,O
1,1,Viscosity of water,L,625,viscosity of water,O,2553,0.000000,L,L
2,2,measurement errors,O,607,measurement errors,L,2557,0.000000,O,O
3,3,Randomness,P,2588,Randomness,O,2832,0.000000,O,O
4,4,Friction,L,1261,Friction,O,266,0.000000,L,L
...,...,...,...,...,...,...,...,...,...,...
664,664,slit isn't perfect,L,2863,the size of the slit,O,834,0.149271,L,L
665,665,the slope that the ball rolls down,L,1875,The slope of the triangle the ball is sitting ...,O,1383,0.149408,L,L
666,666,The slope of the triangle the ball is sitting ...,O,1383,the slope that the ball rolls down,L,1875,0.149408,L,L
667,667,difference in method of recording measurements,L,1998,Different means of measurement,O,1147,0.149739,L,L


Next, we want to count the amount of pairs that are resolved by the amending.

In [30]:
# Check how many of the pairs were amended
mask = df_duplicates_amended['updated_code1'] == df_duplicates_amended['updated_code2']

num_amended = mask.sum()
print(num_amended)

531


It turns out that 531 out of the 669 pairs are resolved(78.9%).

The next step is to implement the changes in the original dataset 'sources'.

In [51]:
# Make a new column in the df for the updated codes
df['amended_code'] = df['code']

# Update the original dataset sources with the amended codes
for i in range(len(df_duplicates_amended)):
    if df_duplicates_amended['code1'][i] != df_duplicates_amended['updated_code1'][i]:
        df.loc[df_duplicates_amended['index1'][i], 'amended_code'] = df_duplicates_amended['updated_code1'][i]
    if df_duplicates_amended['code2'][i] != df_duplicates_amended['updated_code2'][i]:
        df.loc[df_duplicates_amended['index2'][i], 'amended_code'] = df_duplicates_amended['updated_code2'][i]

# Count how many codes were changed
num_changed = (df['code'] != df['amended_code']).sum()
print(num_changed)

153


In fact only 153 responses were reclassified, however, one response could be part of many pairs. When changing the code for a single response, many pairs with this response could be resolved, making the workload less for the qualitative researcher.

Finally, the finished and amended dataset is saved for analysis.

In [ ]:
# Save the amended dataset
df.to_excel('data/sources_amended.xlsx', index=False)